# Refresh Semantic Models

Triggers a refresh for semantic models in a Fabric workspace after data ingestion.
Can be called from a Fabric Data Pipeline (Notebook activity) or run manually.

**Usage:**
1. Set workspace ID and models to refresh in Cell 1
2. Run all cells
3. The notebook triggers refreshes and polls until complete

**Pipeline integration:** Add this notebook as a Notebook activity after your
data ingestion steps. Pass `WORKSPACE_ID` and `MODEL_NAMES` as pipeline parameters.

## 1. Configuration

In [ ]:
# === CONFIGURATION ===
# This cell is tagged 'parameters' so Fabric Data Pipelines can inject values

WORKSPACE_ID = "68c48a5c-8b88-4559-962d-5dc7f29ab96a"  # Target workspace

# Models to refresh — set to None to refresh ALL semantic models in the workspace
MODEL_NAMES = None  # e.g., ["Finance Summary", "SalesAnalyticsModel"]

# Exclude models whose names start with any of these prefixes (case-sensitive)
EXCLUDE_PREFIXES = ["BACKUP_"]  # e.g., ["BACKUP_", "OLD_", "ARCHIVE_"]

# Refresh type: "Full", "Automatic", "DataOnly", "Calculate", "ClearValues"
REFRESH_TYPE = "Full"

# Wait for refresh to complete? If False, fire-and-forget
WAIT_FOR_COMPLETION = True

# Max time (seconds) to wait per model before timing out
MAX_WAIT_SECONDS = 1800  # 30 minutes

# Polling interval (seconds)
POLL_INTERVAL = 30

# Fail notebook if any refresh fails?
FAIL_ON_ERROR = True

# Email notification on failure — set to None to disable
NOTIFY_EMAIL = "data-team@company.com"  # e.g., "admin@company.com"

# SMTP settings (only needed when running locally, not in Fabric notebooks)
SMTP_SERVER = "smtp.office365.com"
SMTP_PORT = 587
SMTP_FROM = "fabric-alerts@company.com"
SMTP_PASSWORD_ENV = "SMTP_PASSWORD"  # env var name holding the password

print(f"Workspace: {WORKSPACE_ID}")
print(f"Models: {'ALL' if MODEL_NAMES is None else MODEL_NAMES}")
print(f"Excluded prefixes: {EXCLUDE_PREFIXES}")
print(f"Refresh type: {REFRESH_TYPE}")
print(f"Wait for completion: {WAIT_FOR_COMPLETION}")
print(f"Notify on failure: {NOTIFY_EMAIL or 'disabled'}")

## 2. Authentication

In [ ]:
import requests
import json
import time
import os
from datetime import datetime

try:
    from notebookutils import mssparkutils
    access_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    IN_FABRIC = True
    print("\u2713 Authenticated via Fabric notebook token")
except ImportError:
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential()
    token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
    access_token = token.token
    IN_FABRIC = False
    print("\u2713 Authenticated via DefaultAzureCredential")

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

PBI_API = "https://api.powerbi.com/v1.0/myorg"

def api_get(url):
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.ok:
        return resp.json()
    print(f"  \u2717 GET {resp.status_code}: {resp.text[:300]}")
    return None

def api_post(url, body=None):
    return requests.post(url, headers=headers, json=body, timeout=60)

def send_failure_email(subject, body_text):
    """Send failure notification email. Uses Fabric notebookutils if available, else SMTP."""
    if not NOTIFY_EMAIL:
        return
    print(f"\nSending failure notification to {NOTIFY_EMAIL}...")
    try:
        if IN_FABRIC:
            # Fabric notebook built-in email via notebookutils
            mssparkutils.notification.sendEmail(
                to=NOTIFY_EMAIL,
                subject=subject,
                body=body_text
            )
            print(f"  \u2713 Email sent via Fabric notebookutils")
        else:
            # Fallback: SMTP (for local/non-Fabric runs)
            import smtplib
            from email.mime.text import MIMEText
            smtp_password = os.environ.get(SMTP_PASSWORD_ENV, "")
            if not smtp_password:
                print(f"  \u26a0\ufe0f SMTP password env var '{SMTP_PASSWORD_ENV}' not set — skipping email")
                return
            msg = MIMEText(body_text)
            msg["Subject"] = subject
            msg["From"] = SMTP_FROM
            msg["To"] = NOTIFY_EMAIL
            with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
                server.starttls()
                server.login(SMTP_FROM, smtp_password)
                server.send_message(msg)
            print(f"  \u2713 Email sent via SMTP")
    except Exception as e:
        print(f"  \u26a0\ufe0f Could not send email: {e}")

ws_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}")
ws_name = ws_info.get("name", "unknown") if ws_info else "unknown"
if ws_info:
    print(f"\u2713 Connected to workspace: {ws_name}")

## 3. Discover semantic models

In [ ]:
# List all datasets (semantic models) in the workspace
datasets_data = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets")
all_models = datasets_data.get("value", []) if datasets_data else []

# Filter out system-generated models (e.g., auto-generated by reports)
user_models = [m for m in all_models if not m.get("name", "").startswith("ModelDefinition")]

# Exclude models matching EXCLUDE_PREFIXES
excluded = [m for m in user_models if any(m["name"].startswith(p) for p in EXCLUDE_PREFIXES)]
user_models = [m for m in user_models if not any(m["name"].startswith(p) for p in EXCLUDE_PREFIXES)]

if excluded:
    print(f"Excluded by prefix ({len(excluded)}):")
    for m in excluded:
        print(f"  [EXCLUDED] {m['name']}")
    print()

# Apply name filter if specified
if MODEL_NAMES is not None:
    target_models = [m for m in user_models if m["name"] in MODEL_NAMES]
    not_found = set(MODEL_NAMES) - {m["name"] for m in target_models}
    if not_found:
        print(f"\u26a0\ufe0f Models not found in workspace: {sorted(not_found)}")
else:
    target_models = user_models

print(f"Semantic models to refresh ({len(target_models)}):\n")
for m in sorted(target_models, key=lambda x: x["name"]):
    configured_by = m.get("configuredBy", "unknown")
    print(f"  {m['name']}  (id={m['id']}, configuredBy={configured_by})")

if not target_models:
    print("\nNo models to refresh — stopping.")

## 4. Trigger refreshes

In [ ]:
# Trigger refresh for each model
refresh_status = {}  # model_name -> {"id": dataset_id, "triggered": bool, "status": str}

for model in target_models:
    model_name = model["name"]
    model_id = model["id"]
    print(f"\nTriggering {REFRESH_TYPE} refresh for '{model_name}'...")

    payload = {
        "type": REFRESH_TYPE,
        "notifyOption": "NoNotification"
    }
    resp = api_post(
        f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{model_id}/refreshes",
        body=payload
    )

    if resp.status_code == 202:
        print(f"  \u2713 Refresh queued (202 Accepted)")
        refresh_status[model_name] = {"id": model_id, "triggered": True, "status": "Unknown"}
    else:
        error_text = resp.text[:300]
        print(f"  \u2717 Failed ({resp.status_code}): {error_text}")
        refresh_status[model_name] = {"id": model_id, "triggered": False, "status": f"TriggerFailed: {resp.status_code}"}

triggered_count = sum(1 for v in refresh_status.values() if v["triggered"])
print(f"\n{triggered_count}/{len(target_models)} refresh(es) triggered")

## 5. Wait for completion (optional)

In [ ]:
if not WAIT_FOR_COMPLETION:
    print("WAIT_FOR_COMPLETION = False — skipping. Check refresh status in Fabric portal.")
else:
    # Poll each model's latest refresh until complete
    pending = {name: info for name, info in refresh_status.items() if info["triggered"]}
    start_time = time.time()

    while pending:
        elapsed = time.time() - start_time
        if elapsed > MAX_WAIT_SECONDS:
            print(f"\n\u26a0\ufe0f Timeout after {MAX_WAIT_SECONDS}s — {len(pending)} model(s) still refreshing")
            for name in pending:
                refresh_status[name]["status"] = "Timeout"
            break

        time.sleep(POLL_INTERVAL)
        print(f"\n[{int(elapsed)}s] Checking {len(pending)} model(s)...")

        still_pending = {}
        for name, info in pending.items():
            # Get the most recent refresh entry
            history = api_get(
                f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{info['id']}/refreshes?$top=1"
            )
            if not history or not history.get("value"):
                still_pending[name] = info
                continue

            latest = history["value"][0]
            status = latest.get("status", "Unknown")

            if status == "Unknown":  # Still in progress
                print(f"  {name}: refreshing...")
                still_pending[name] = info
            elif status == "Completed":
                print(f"  \u2713 {name}: completed")
                refresh_status[name]["status"] = "Completed"
            elif status == "Failed":
                error_msg = latest.get("serviceExceptionJson", "")
                print(f"  \u2717 {name}: FAILED — {error_msg[:200]}")
                refresh_status[name]["status"] = "Failed"
                refresh_status[name]["error"] = error_msg[:500]
            else:
                print(f"  {name}: {status}")
                if status in ("Cancelled", "Disabled"):
                    refresh_status[name]["status"] = status
                else:
                    still_pending[name] = info

        pending = still_pending

    # Summary
    print("\n" + "=" * 50)
    print("REFRESH SUMMARY")
    print("=" * 50)
    for name, info in sorted(refresh_status.items()):
        icon = "\u2713" if info["status"] == "Completed" else "\u2717"
        print(f"  {icon} {name}: {info['status']}")

    # Send email notification if any failures
    failed = [n for n, i in refresh_status.items() if i["status"] not in ("Completed",)]
    if failed and NOTIFY_EMAIL:
        subject = f"Semantic Model Refresh Failed — {ws_name}"
        lines = [
            f"Workspace: {ws_name} ({WORKSPACE_ID})",
            f"Time: {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC",
            f"Refresh type: {REFRESH_TYPE}",
            "",
            f"Failed models ({len(failed)}):",
        ]
        for name in sorted(failed):
            info = refresh_status[name]
            lines.append(f"  \u2717 {name} — {info['status']}")
            if info.get("error"):
                lines.append(f"    Error: {info['error']}")
        lines.append("")
        succeeded = [n for n, i in refresh_status.items() if i["status"] == "Completed"]
        if succeeded:
            lines.append(f"Succeeded ({len(succeeded)}):")
            for name in sorted(succeeded):
                lines.append(f"  \u2713 {name}")
        send_failure_email(subject, "\n".join(lines))

    if failed and FAIL_ON_ERROR:
        raise RuntimeError(f"Refresh failed for: {', '.join(failed)}")